In [35]:
import torch
from torch import nn, optim

def forsaken(f_theta_0, T, lambda_, omega, P, D_f, eta_mu, xi):
    model_t = f_theta_0
    theta_0 = [param.clone().detach() for param in f_theta_0.parameters()]
    mu = [torch.zeros_like(param, requires_grad=True) for param in theta_0]
    criterion = nn.KLDivLoss()
    optimizer = optim.LBFGS(mu, lr=eta_mu)

    for _ in range(T):
        with torch.no_grad():
            for param, theta, m in zip(model_t.parameters(), theta_0, mu):
                param.copy_((theta - xi * m).detach())

        upsilon = model_t(D_f)
        loss = criterion(upsilon, P) + lambda_ * omega * sum(torch.norm(m, p=1) for m in mu)

        loss.backward()
        optimizer.step()

    return model_t



In [73]:
###################################
# 1) Imports
###################################
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [74]:

###################################
# 2) Préparation des données MNIST
###################################
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


In [75]:

###################################
# 3) Définition d'un modèle FC simple
###################################
class FullyConnectedNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = FullyConnectedNN()
model2 = FullyConnectedNN()

In [76]:

###################################
# 4) Entraînement rapide (optionnel)
###################################
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 2  # juste pour illustrer
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")



###################################
# 4) Entraînement rapide (optionnel)
###################################
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 2  # juste pour illustrer
for epoch in range(epochs):
    model2.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")



Epoch 1/2, Loss: 0.8113
Epoch 2/2, Loss: 0.3107
Epoch 1/2, Loss: 0.2552
Epoch 2/2, Loss: 0.2185


In [89]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.stateless import functional_call
import copy

def forsaken(f_theta_0, T, lambda_, omega, P, D_f, eta_mu, xi):
    # On travaille sur le modèle cible
    model_t = copy.deepcopy(f_theta_0)
    # On crée un dictionnaire de paramètres initiaux détachés
    theta_0 = {name: param.clone().detach() for name, param in model_t.named_parameters()}
    # On initialise μ pour chaque paramètre (avec requires_grad=True)
    mu = {name: torch.zeros_like(param, requires_grad=True) for name, param in theta_0.items()}
    
    criterion = nn.KLDivLoss(reduction='batchmean')
    # Optimiseur SGD sur les valeurs de mu
    optimizer = optim.SGD(list(mu.values()), lr=eta_mu)
    
    for _ in range(T):
        # Remettre à zéro les gradients
        optimizer.zero_grad()
        
        # Calculer les nouveaux paramètres de manière différentiable
        new_params = {name: theta_0[name] - xi * mu[name] for name in theta_0}
        
        # Effectuer la forward pass avec ces nouveaux paramètres
        upsilon = functional_call(model_t, new_params, D_f)
        log_probs = F.log_softmax(upsilon, dim=1)
        
        # Calculer la perte
        loss = criterion(log_probs, P) + lambda_ * omega * sum(torch.norm(m, p=1) for m in mu.values())
        
        # Backward pour calculer les gradients de mu
        loss.backward()
        
        # Mettre à jour les valeurs de mu avec SGD
        optimizer.step()
        
        print(f"Loss: {loss.item()}")

    # Après optimisation, mettre à jour les paramètres du modèle
    final_params = {name: theta_0[name] - xi * mu[name].detach() for name in theta_0}
    for name, param in model_t.named_parameters():
        param.data.copy_(final_params[name])
    
    return model_t


In [77]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.stateless import functional_call
import copy

def forsaken(f_theta_0, T, lambda_, omega, P, D_f, eta_mu, xi):
    # On travaille sur le modèle cible
    model_t = copy.deepcopy(f_theta_0)
    # On crée un dictionnaire de paramètres initiaux détachés
    theta_0 = {name: param.clone().detach() for name, param in model_t.named_parameters()}
    # On initialise μ pour chaque paramètre (avec requires_grad=True)
    mu = {name: torch.zeros_like(param, requires_grad=True) for name, param in theta_0.items()}
    
    criterion = nn.KLDivLoss(reduction='batchmean')
    # Optimiseur LBFGS sur les valeurs de mu
    optimizer = optim.LBFGS(list(mu.values()), lr=eta_mu)
    
    def closure():
        optimizer.zero_grad()
        # Calculer les nouveaux paramètres de manière différentiable
        new_params = {name: theta_0[name] - xi * mu[name] for name in theta_0}
        # Effectuer la forward pass avec ces nouveaux paramètres
        upsilon = functional_call(model_t, new_params, D_f)
        log_probs = F.log_softmax(upsilon, dim=1)
        loss = criterion(log_probs, P) + lambda_ * omega * sum(torch.norm(m, p=1) for m in mu.values())
        loss.backward()
        print(f"Loss: {loss.item()}")
        return loss

    for _ in range(T):
        optimizer.step(closure)
    
    # Après optimisation, mettre à jour les paramètres du modèle
    final_params = {name: theta_0[name] - xi * mu[name].detach() for name in theta_0}
    for name, param in model_t.named_parameters():
        param.data.copy_(final_params[name])
    
    return model_t


In [78]:
import random
from torch.utils.data import Subset, DataLoader

# Sélectionner 100 indices aléatoires depuis train_dataset
forget_indices = random.sample(range(len(train_dataset)), 100)

# Créer le sous-ensemble D_f
D_f = Subset(train_dataset, forget_indices)
print(f"Nombre de points dans D_f à forget : {len(D_f)}")

# Optionnel : créer un DataLoader pour D_f (pour l'évaluation)
forget_loader = DataLoader(D_f, batch_size=64, shuffle=False)


Nombre de points dans D_f à forget : 100


In [91]:
import random
from torch.utils.data import Subset, DataLoader
import torch.nn.functional as F

# Supposons que train_dataset et model (votre modèle initial) soient déjà définis

# 1. Création d'un sous-ensemble D_f (par exemple, 100 points aléatoires)
forget_indices = random.sample(range(len(train_dataset)), 100)
D_f = Subset(train_dataset, forget_indices)
forget_loader = DataLoader(D_f, batch_size=64, shuffle=False)
print(f"Nombre de points dans D_f à forget : {len(D_f)}")

# 2. On prend un batch du DataLoader de D_f pour passer à forsaken
images_forget, labels_forget = next(iter(forget_loader))

# 3. Création de la distribution cible P (uniforme) pour chaque image du batch
P = torch.full((images_forget.size(0), 10), 1/10)

# 4. Définition des hyperparamètres pour forsaken
T = 100       # Nombre d'itérations externes
lambda_ = 1.0
omega = 1.0
eta_mu = 0.1
xi = 0.5

# 5. Appel de la fonction forsaken pour obtenir le modèle modifié
model_forsaken = forsaken(
    f_theta_0=model,
    T=T,
    lambda_=0,
    omega=omega,
    P=P,
    D_f=images_forget,
    eta_mu=eta_mu,
    xi=xi
)
print("Fin de la procédure Forsaken.")

# 6. Évaluation de model_forsaken sur l'ensemble D_f
def evaluate_model_on_forget(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    return 100 * correct / total

accuracy_forget = evaluate_model_on_forget(model_forsaken, forget_loader)
print(f"Accuracy sur D_f après Forsaken : {accuracy_forget:.2f}%")

accuracy_base = evaluate_model_on_forget(model, forget_loader)
print(f"Accuracy sur D_f après Forsaken : {accuracy_base:.2f}%")


Nombre de points dans D_f à forget : 100
Loss: 5.699652194976807
Loss: 4.471652984619141
Loss: 3.5875611305236816
Loss: 2.885462522506714
Loss: 2.318739652633667
Loss: 1.854866623878479
Loss: 1.479772686958313
Loss: 1.182206153869629
Loss: 0.9501779079437256
Loss: 0.7748945355415344
Loss: 0.642569363117218
Loss: 0.5415499806404114
Loss: 0.46250781416893005
Loss: 0.39985018968582153
Loss: 0.34927427768707275
Loss: 0.30837705731391907
Loss: 0.275283545255661
Loss: 0.24810004234313965
Loss: 0.22518785297870636
Loss: 0.20585660636425018
Loss: 0.18932214379310608
Loss: 0.174984410405159
Loss: 0.16259068250656128
Loss: 0.15170514583587646
Loss: 0.14203990995883942
Loss: 0.1334395408630371
Loss: 0.12575766444206238


c:\Users\alici\anaconda3\lib\site-packages\torch\nn\utils\stateless.py:214: UserWarning: This API is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use torch.func.functional_call instead which is a drop-in replacement for this API.
  warnings.warn(


Loss: 0.11882826685905457
Loss: 0.11257348209619522
Loss: 0.10684317350387573
Loss: 0.10159386694431305
Loss: 0.09682246297597885
Loss: 0.09242702275514603
Loss: 0.08840147405862808
Loss: 0.08466382324695587
Loss: 0.08118163794279099
Loss: 0.07796687632799149
Loss: 0.07496752589941025
Loss: 0.0721484124660492
Loss: 0.0695180743932724
Loss: 0.06705465167760849
Loss: 0.0647483542561531
Loss: 0.06257781386375427
Loss: 0.06052997708320618
Loss: 0.058592140674591064
Loss: 0.056757379323244095
Loss: 0.05501210689544678
Loss: 0.05335875228047371
Loss: 0.05178895592689514
Loss: 0.050290290266275406
Loss: 0.04885868728160858
Loss: 0.047490425407886505
Loss: 0.0461883544921875
Loss: 0.04495181888341904
Loss: 0.04376981034874916
Loss: 0.04263971373438835
Loss: 0.04155607894062996
Loss: 0.04051772132515907
Loss: 0.039524342864751816
Loss: 0.038575634360313416
Loss: 0.03766286373138428
Loss: 0.03678125888109207
Loss: 0.03593193739652634
Loss: 0.0351201593875885
Loss: 0.034339770674705505
Loss: 0.03